# AGENTIC AI
## SESSION 1

Assignment questions and answers


## Question 1

Evaluate prompt engineering, RAG, and fine-tuning for a
food delivery chatbot recommending dishes using dietary restrictions,
order history, and real-time menu availability. Recommend a suitable
approach or combination and explain why the alternatives are less
suitable.

**Answer:**

### Evaluation of Approaches

| Approach | Dietary Restrictions | Order History | Real-time Menu Availability | Suitability |
| :--- | :--- | :--- | :--- | :--- |
| **Prompt Engineering** | Good via system instructions. | Limited by context window size. | Poor; cannot access live external data alone. | Insufficient on its own. |
| **RAG (Retrieval-Augmented Generation)** | Excellent; retrieves user profile data. | Excellent; fetches past orders from databases. | Excellent; queries live inventory and menu databases. | Highly suitable. |
| **Fine-Tuning** | Learns general patterns, not dynamic user data. | Cannot update fast enough for new orders. | Fails; model weights are static and cannot reflect live stock. | Unsuitable for dynamic data. |

---

### Recommended Approach
**A combination of RAG and Prompt Engineering.**

* **RAG** queries real-time databases to fetch the live restaurant menu, current stock, the user's saved dietary restrictions, and recent order history.
* **Prompt Engineering** structures the system prompt to enforce tone, format the final response, and instruct the LLM to strictly recommend dishes matching the retrieved constraints.

---

### Why the Alternatives are Less Suitable

* **Fine-Tuning alone or combined:** Model weights are static. Updating it to reflect real-time dish availability or a new order would require continuous, expensive retraining, and it causes hallucinations regarding live inventory.
* **Prompt Engineering alone:** Cannot access external or changing data (such as fluctuating restaurant stock and growing order histories) without a retrieval mechanism.

## Question 2

Explain what LoRA modifies in a pre-trained model compared with full
    fine-tuning, why it uses less VRAM, and what trade-off is accepted
    when using it on a consumer GPU with 8 GB VRAM.

**Answer:**

| Feature | Full Fine-Tuning | LoRA (Low-Rank Adaptation) |
| :--- | :--- | :--- |
| **What it modifies** | Updates all original pre-trained weight matrices directly. | Freezes original weights and adds small, trainable low-rank decomposition matrices alongside them. |

* **Why it uses less VRAM:**
  * **Fewer trainable parameters:** Drastically reduces the memory needed to store optimizer states (e.g., Adam states) and weight gradients.
  * **Reduced activation memory:** Freezing the base model allows for optimizations that lower memory overhead during the backward pass.

* **Trade-off accepted on an 8 GB GPU:**
  * **Capacity vs. Stability:** You sacrifice a small amount of model adaptation capacity (expressiveness) and must use small batch sizes, low rank values ($r$), or lower precision (QLoRA) to avoid Out-Of-Memory (OOM) errors, potentially increasing training time.

## Question 3

A fine-tuned complaint classifier performs well on common complaints
    but fails on rare complaints absent from its 6,000 labelled training
    examples. Identify two fundamental limitations of fine-tuning. For
    each, recommend re-fine-tuning with more data, RAG, or few-shot
    prompting, and state the condition under which the remedy is valid.

**Answer:**

1. **Limitation: Inability to generalize to out-of-distribution or unseen classes**
   * **Recommendation:** Few-shot prompting.
   * **Condition:** Valid when the base model already possesses general semantic understanding of the rare complaints, and the new categories can be clearly defined within the model's context window.

2. **Limitation: Static internal parametric memory (knowledge staleness)**
   * **Recommendation:** Retrieval-Augmented Generation (RAG).
   * **Condition:** Valid when there is a constantly updating, external knowledge base or labeled archive of rare complaint examples that can be dynamically retrieved at query time.

## Question 4

Trace the request “Order 2 portions of Masala Dosa from Saravana
    Bhavan for delivery at 7 PM tonight” using the Perceive → Think →
    Act loop. Explain what is perceived, what reasoning and ambiguity
    resolution occur, which tools are called and in what order, and how
    the loop continues if the restaurant is unavailable at 7 PM.

**Answer:**

### 1. Perceive
The agent receives and parses the user input:
* **Item & Quantity:** 2 portions of Masala Dosa
* **Restaurant:** Saravana Bhavan
* **Fulfillment Type:** Delivery
* **Target Time:** 7:00 PM tonight

### 2. Think (Reasoning & Ambiguity Resolution)
* **Ambiguity Resolution:** Identify the exact Saravana Bhavan branch nearest to the user's saved address. Map "tonight" to the current calendar date. 
* **State Check:** Determine if the restaurant accepts scheduled orders and if the item is available.
* **Plan:** 
  1. Fetch user location.
  2. Search for the restaurant branch.
  3. Verify item availability and price.
  4. Schedule delivery for 7:00 PM.
  5. Process payment/confirmation.

### 3. Act (Tool Calls in Order)
1. `getUserLocation()`: Retrieves the user's default delivery address.
2. `searchRestaurant(name="Saravana Bhavan", location=user_location)`: Finds the nearest eligible branch.
3. `checkMenuAndAvailability(restaurant_id, item="Masala Dosa", quantity=2, time="19:00")`: Confirms item availability and whether the 7:00 PM slot is open.
4. `scheduleOrder(restaurant_id, items=[...], delivery_time="19:00")`: Places the scheduled order.

---

### Continuation if Unavailable at 7 PM

* **Perceive:** The `checkMenuAndAvailability` tool returns a failure status (e.g., `"Status: Closed / No delivery slots at 19:00"`).
* **Think:** 
  * The primary goal cannot be fulfilled as requested.
  * Formulate alternative options: Find the earliest available slot at Saravana Bhavan, or look for Masala Dosa at an alternative top-rated restaurant for 7:00 PM.
  * Decide to prompt the user for a decision rather than guessing.
* **Act:** Call `sendMessageToUser()` to report the issue: *"Saravana Bhavan cannot deliver at 7:00 PM. Would you like to schedule it for the next available time (e.g., 7:45 PM), or order from a different restaurant for 7:00 PM?"*
* **Loop Resumes:** The agent awaits the user's response to enter a new **Perceive → Think → Act** cycle.

## Question 5

Design a Database MCP Server for food-delivery support and
    analytics. Name the tools to expose (e.g. list_tables,
    describe_table, run_select_query), choose read-only or read-write
    access and justify it, give a realistic delayed-order question, and
    explain a safeguard that prevents unnecessary sensitive data from
    being returned.

**Answer:**

* **Tools to Expose:**
  * `list_tables`: Lists all available tables in the database.
  * `describe_table`: Returns the schema, column names, and data types for a specified table.
  * `run_select_query`: Executes a read-only SQL query and returns the results.

* **Access Level & Justification:**
  * **Read-only access.** Support and analytics tasks only require viewing data to resolve issues and generate insights; restricting write access prevents accidental data modification, data corruption, and unauthorized state changes.

* **Realistic Delayed-Order Question:**
  * "Why is order #84920 delayed, and what is the current status and location of the assigned driver?"

* **Safeguard Against Sensitive Data:**
  * **Column-level filtering / Column masking:** The server enforces an allowlist or masks PII columns (such as passwords, credit card numbers, and customer phone numbers), automatically excluding them from query results before returning data to the model.

## Question 6

For a food-delivery support agent using the ReAct pattern, handle:
    “My order #FD4521 was 40 minutes late and the food arrived cold. I
    want either a full refund or a significant discount on my next
    order.” Walk through at least two complete Reason → Action →
    Observation cycles, stating the reasoning, tool action, and
    observation in each. Explain why separating reasoning from action
    matters for financial decisions.

**Answer:**

**Cycle 1**
* **Reason:** The user reported order #FD4521 was 40 minutes late and cold, demanding a refund or discount. First, I need to verify the order status, delivery timestamp, and eligibility criteria in the system.
* **Action:** `check_order_details(order_id="FD4521")`
* **Observation:** Order #FD4521 confirmed delivered 42 minutes past the estimated delivery time. Total paid: $35.00. Account is in good standing with no prior refund abuse.

**Cycle 2**
* **Reason:** The delay exceeds the 30-minute threshold for service failures. Company policy allows a full refund or a 50% discount voucher. The customer requested a full refund or a significant discount; issuing a full refund directly resolves the complaint under policy limits.
* **Action:** `issue_refund(order_id="FD4521", amount=35.00, reason="Late delivery and cold food")`
* **Observation:** Refund of $35.00 successfully processed to the original payment method. Transaction ID: #RF99812.

***

**Why Separating Reasoning from Action Matters for Financial Decisions**
Separating reasoning from action ensures the agent validates business rules, checks eligibility, and confirms limits before executing irreversible financial transactions. This deliberate step prevents costly errors, hallucinations, or unauthorized payouts by creating an auditable trace of the logic that justified the money movement.